For an OLS regression using `statsmodels`, you can fit the model and then perform the standard regression diagnostics as follows.

### **1. Fit the OLS Model**

In [ ]:
import statsmodels.api as sm
# Add intercept
X_sm = sm.add_constant(X)
# Fit OLS
ols_model = sm.OLS(y, X_sm).fit()
# Model summary
print(ols_model.summary())

In [ ]:
# 1. Normality of the Residuals

### (a) Histogram
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.hist(ols_model.resid, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Histogram of Residuals")
plt.show()

### (b) Q-Q Plot
import statsmodels.api as sm

sm.qqplot(ols_model.resid, line='45', fit=True)
plt.title("Q-Q Plot of Residuals")
plt.show()

### (c) Shapiro-Wilk Test

from scipy.stats import shapiro

stat, p = shapiro(ols_model.resid)

print("Shapiro-Wilk Statistic:", stat)
print("p-value:", p)

if p > 0.05:
    print("Residuals are normally distributed.")
else:
    print("Residuals are NOT normally distributed.")

### (d) Jarque-Bera Test

from statsmodels.stats.stattools import jarque_bera

jb_stat, jb_pvalue, skew, kurtosis = jarque_bera(ols_model.resid)

print("Jarque-Bera Statistic:", jb_stat)
print("p-value:", jb_pvalue)

# 2. Multicollinearity

### Variance Inflation Factor (VIF)
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame({
    "Variable": X_sm.columns,
    "VIF": [variance_inflation_factor(X_sm.values, i)
            for i in range(X_sm.shape[1])]
})

print(vif)

Interpretation

* VIF = 1 → No multicollinearity
* 1–5 → Acceptable
* 5–10 → Moderate multicollinearity
* > 10 → Serious multicollinearity

---

# 3. Heteroscedasticity

### Breusch-Pagan Test

```python
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test = het_breuschpagan(ols_model.resid, ols_model.model.exog)

labels = [
    "LM Statistic",
    "LM-Test p-value",
    "F-Statistic",
    "F-Test p-value"
]

print(pd.Series(bp_test, index=labels))
```

Interpretation

* p > 0.05 → Homoscedasticity (assumption met)
* p < 0.05 → Heteroscedasticity present

---

### White Test

```python
from statsmodels.stats.diagnostic import het_white

white_test = het_white(ols_model.resid, ols_model.model.exog)

labels = [
    "LM Statistic",
    "LM-Test p-value",
    "F-Statistic",
    "F-Test p-value"
]

print(pd.Series(white_test, index=labels))
```

---

### Residuals vs Fitted Plot

```python
plt.figure(figsize=(6,4))
plt.scatter(ols_model.fittedvalues, ols_model.resid)
plt.axhline(y=0, linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")
plt.show()
```

---

# 4. Autocorrelation

### Durbin-Watson Test

```python
from statsmodels.stats.stattools import durbin_watson

dw = durbin_watson(ols_model.resid)

print("Durbin-Watson Statistic:", dw)
```

Interpretation

* ≈2 → No autocorrelation
* <2 → Positive autocorrelation
* > 2 → Negative autocorrelation

---

### Breusch-Godfrey Test

```python
from statsmodels.stats.diagnostic import acorr_breusch_godfrey

bg_test = acorr_breusch_godfrey(ols_model, nlags=5)

labels = [
    "LM Statistic",
    "LM-Test p-value",
    "F-Statistic",
    "F-Test p-value"
]

print(pd.Series(bg_test, index=labels))
```

Interpretation

* p > 0.05 → No autocorrelation
* p < 0.05 → Autocorrelation exists

---

## Complete Diagnostic Summary

| Assumption         | Test                       | Decision Rule                   |
| ------------------ | -------------------------- | ------------------------------- |
| Normality          | Shapiro-Wilk / Jarque-Bera | p > 0.05 → Normal residuals     |
| Multicollinearity  | VIF                        | VIF < 5 (or <10) desirable      |
| Heteroscedasticity | Breusch-Pagan / White      | p > 0.05 → Homoscedastic        |
| Autocorrelation    | Durbin-Watson              | ≈2 indicates no autocorrelation |
| Autocorrelation    | Breusch-Godfrey            | p > 0.05 → No autocorrelation   |

These tests constitute a comprehensive set of OLS diagnostic checks commonly reported in regression analyses.
